In [5]:
import geopandas as gdp
import pandas as pd
from shapely import wkt
from pyarrow import Table
from shapely.geometry import Point

In [ ]:
# Assuming geometry is in WKT format and stored in a column named 'geometry'
def wkt_to_wkb(table):
    wkb_geometry = [wkt.loads(geom).wkb if geom is not None else None for geom in table['geometry'].to_pylist()]
    return table.replace_column(table.schema.get_field_index('geometry'), Table.from_arrays([wkb_geometry], names=['geometry']))

In [2]:
df = pd.read_parquet(
    "s3://weave.energy/smart-meter.parquet", 
    # The 7pm evening peak on the first day all DNOs have data for
    filters=[("data_collection_log_timestamp", "==", pd.Timestamp("2024-2-12 19:00Z"))])

In [14]:
gdf = gdp.GeoDataFrame(
                df,
                geometry=[Point(geo_dict['x'], geo_dict['y']) for geo_dict in df['geometry']],
                crs="EPSG:4326"
)

In [15]:
gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 104502 entries, 0 to 104501
Data columns (total 9 columns):
 #   Column                           Non-Null Count   Dtype              
---  ------                           --------------   -----              
 0   dataset_id                       104502 non-null  object             
 1   dno_alias                        104502 non-null  object             
 2   aggregated_device_count_active   104502 non-null  int64              
 3   total_consumption_active_import  104502 non-null  int64              
 4   data_collection_log_timestamp    104502 non-null  datetime64[ns, UTC]
 5   geometry                         104502 non-null  geometry           
 6   secondary_substation_unique_id   104502 non-null  object             
 7   lv_feeder_unique_id              104502 non-null  object             
 8   bbox                             104502 non-null  object             
dtypes: datetime64[ns, UTC](1), geometry(1), int64(2), ob